In [ ]:
import os
import re
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.models import Model

In [ ]:
#Load and pre process captions
def load_captions(filename):
    captions = {}
    with open(filename, 'r') as f:
        next(f)  # skip header
        for line in f:
            parts = line.strip().split(',', 1)
            if len(parts) < 2:
                continue
            img, caption = parts
            caption = re.sub(r'[^a-zA-Z ]', '', caption.lower()).strip()
            captions.setdefault(img, []).append(caption)
    return captions

captions_path = 'captions.txt' #Path to captions text file
captions_dict = load_captions(captions_path)
print("✅ Captions loaded:", len(captions_dict))

In [ ]:
#Tokenize Captions
all_captions = [c for lst in captions_dict.values() for c in lst]
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index) + 1
max_length = 34  # fixed for Flickr8k dataset

In [ ]:
#Extract features from Inception V3
img_folder = 'Images' #Path to Images folder
features_path = 'features.pkl' #Path to captured features file

if os.path.exists(features_path):
    #Load pre-saved features
    with open(features_path, 'rb') as f:
        features = pickle.load(f)
    print("Loaded features from file:", len(features))
else:
    #Extract features using InceptionV3
    model_cnn = InceptionV3(weights='imagenet', include_top=False, pooling='avg')
    features = {}
    for img_name in os.listdir(img_folder):
        path = os.path.join(img_folder, img_name)
        image = load_img(path, target_size=(299, 299))
        image = img_to_array(image)
        image = np.expand_dims(image, axis=0)
        image = preprocess_input(image)
        feature = model_cnn.predict(image, verbose=0)
        features[img_name] = feature
    #Save features for later use
    with open(features_path, 'wb') as f:
        pickle.dump(features, f)
    print("Extracted & saved features:", len(features))

In [ ]:
#Build the model
img_input = Input(shape=(2048,))
img_layer = Dropout(0.5)(img_input)
img_layer = Dense(256, activation='relu')(img_layer)

txt_input = Input(shape=(max_length,))
txt_layer = Embedding(vocab_size, 256, mask_zero=True)(txt_input)
txt_layer = Dropout(0.5)(txt_layer)
txt_layer = LSTM(256)(txt_layer)

decoder = add([img_layer, txt_layer])
decoder = Dense(256, activation='relu')(decoder)
output = Dense(vocab_size, activation='softmax')(decoder)

model = Model(inputs=[img_input, txt_input], outputs=output)
model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

In [ ]:
#Prepare Data for training
def create_sequences(tokenizer, max_length, captions_list, photo_feature, vocab_size):
    X1, X2, y = [], [], []
    for caption in captions_list:
        seq = tokenizer.texts_to_sequences([caption])[0]
        for i in range(1, len(seq)):
            in_seq, out_seq = seq[:i], seq[i]
            in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
            out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
            X1.append(photo_feature)
            X2.append(in_seq)
            y.append(out_seq)
    return np.array(X1), np.array(X2), np.array(y)

In [ ]:
#Data Generator
def data_generator(captions_dict, features, tokenizer, max_length, vocab_size):
    while True:
        for img_name, captions in captions_dict.items():
            if img_name not in features:
                continue
            photo_feature = features[img_name][0]
            X1, X2, y = create_sequences(tokenizer, max_length, captions, photo_feature, vocab_size)
            yield [X1, X2], y

In [ ]:
#Train the model
steps = len(captions_dict)

generator = data_generator(captions_dict, features, tokenizer, max_length, vocab_size)

model.fit(generator, epochs=10, steps_per_epoch=steps, verbose=1)